# Assigment 4

This assigment will be graded if everything works well. I will run the script as once and everything should be done without errors and mistakes. I should be able to run your scripts in my computer and get all the results. **USE RELATIVE PATHS**. An error or exception or anything that breaks the code will means NO GRADE (0). Additionally, you are not able to modify any file handly. It also means NO GRADE (0). Comment everything you think will help others read your script. We expect 0 errors using GitHub. Everything will be graded!

**ASK EVERYTHING! WE ARE HERE TO HELP YOU!**

In this path **..\_data\sbs\B_RawData\bancos** you will find scraped data from [this link](https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIActivaTipoCreditoEmpresa.aspx?tip=B). We get all the information of the last available day of every each month.

1. Generate a function called **import_data_sbs_banks** that take as arguments `start_month`, `start_year`, `end_month` and `end_year`. This function should be able to import the files in **..\_data\sbs\B_RawData\bancos** taking into consideration these arguments. Additionally, it should work if it is only given the first two arguments `(start_mont, start_year)`. For example, this script `import_data_sbs_banks(10, 2015)` should return only the data for October 2015. The function should `returns` a nested dictionary. The `main key` must be year and `second key` must be `month`. The values should be the dataframes. 

In [ ]:
# Importando librerias
import os
import pandas as pd
from datetime import datetime
import glob # Para las coincidencias en el nombre del archivo a importar

In [ ]:
def import_data_sbs_banks(start_month : int = None, start_year : int = None, end_month : int = None, end_year : int = None):
    # Restricción para mes faltante
    if start_month is None:
        raise TypeError("Falta mes de la fecha de inicio.")

    # Restricción para año faltante
    if start_year is None:
        raise TypeError("Falta año de la fecha de inicio.")

    # Restricción para mes inicial no entero
    if not isinstance(start_month, int):
        #raise TypeError("start_month no es entero.")
        start_month = int(start_month)

    # Restricción para año inicial no entero
    if not isinstance(start_year, int):
        #raise TypeError("start_year no es entero.")
        start_year = int(start_year)

    # Restricción para mes final no entero
    if end_month is not None:
        if not isinstance(end_month, int):
            #raise TypeError("end_month no es entero.")
            end_month = int(end_month)

    # Restricción para año final no entero
    if end_year is not None:
        if not isinstance(end_year, int):
            #raise TypeError("end_year no es entero.")
            end_year = int(end_year)

    # Restricción para mes inicial menor a cero
    if start_month <= 0:
        raise TypeError("star_month es menor o igual a cero.")

    # Restricción para mes inicial mayor a 12
    if start_month > 12:
        raise TypeError("El mes de inicio no puede ser mayor a 12.")

    # Restricción para año inicial menor a cero
    if start_year <= 0:
        raise TypeError("start_year es menor o igual a cero.")

    # Restricción para mes final menor a cero
    if end_month is not None:
        if end_month <= 0:
            raise TypeError("end_month es menor o igual a cero.")

    # Restricción para mes final mayor a 12
    if end_month is not None:
        if end_month > 12:
            raise TypeError("El mes del final no puede ser mayor a 12.")

    # Restricción para año final menor a cero
    if end_year is not None:
        if end_year <= 0:
            raise TypeError("end_year es menor o igual a cero.")        

    # Ruta de los archivos
    path = "../../_data/sbs/B_RawData/bancos/"

    # Obteniendo una lista con todas las fechas con información (se cuenta con los archivos excel)
    intervalo = []
    for file in os.listdir(path):
        if file.endswith('.xlsx'):
            x = file.rsplit(".", 1)[0]
            x = x[15:]
            x = x.replace("_", "-")
            x = datetime.strptime(x, "%m-%Y")
            intervalo.append(x)

    # Ordenando la lista de fechas con información
    intervalo.sort()
    min = intervalo[0] # Menor fecha con información
    max = intervalo[-1] # Mayor fecha con información

    # Restricción para solicitud de fechas fuera del intervalo que se tiene información
    start = datetime.strptime(str(start_month) + "-" + str(start_year), "%m-%Y")
    if end_month is None or end_year is None:
        end = start
    else:
        end = datetime.strptime(str(end_month) + "-" + str(end_year), "%m-%Y")
    if start < min or end < min or start > max or end > max:
        raise TypeError("Solicitud de fecha fuera del intervalo")

    # Creando el diccionario que guardará la data importada
    dict = {}

    # Caso 1: Solo se tiene la fecha inicial
    if end_month is None and end_year is None:
        # Restricción cuando la fecha inicial no está completa
        if start_month is None or start_year is None:
            raise TypeError("Fecha incompleta")
        
        # Fecha inicial completa
        else:
            date = datetime.strptime(str(start_month) + "-" + str(start_year), "%m-%Y")
            
            # Restricción para fecha que no se tiene información
            if date not in intervalo:
                raise TypeError("No se tiene informacion de ese mes-año")

            # Importando el df de la fecha inicial
            else:
                # Buscando el archivo con el mes y año
                patron = os.path.join(path, f"table_clean_*_{start_month}_{start_year}.xlsx")

                # Identificando todos los archivos con las coincidencias de mes y año
                archivo = glob.glob(patron)

                # Importando el archivo
                df = pd.read_excel(archivo[0])

                # Creando el key principal y secundario
                dict[start_year] = {}
                dict[start_year][start_month] = df

                return dict

    # Caso 2: Falta mes de la fecha final
    elif end_month is None:
        raise TypeError("Falta mes de la fecha final")

    # Caso 3: Falta año de la fecha final
    elif end_year is None:
        raise TypeError("Falta año de la fecha final")

    # Caso 4: La fecha final es menor a la fecha inicial
    elif datetime.strptime(str(start_month) + "-" + str(start_year), "%m-%Y") > datetime.strptime(str(end_month) + "-" + str(end_year), "%m-%Y"):
        raise TypeError("Fecha final menor a la fecha inicial")

    # Caso 5: Se tiene las fechas inicial y final completas
    else:

        # Identificando las fechas intermedias dentro del intervalo de solicitud
        start = datetime.strptime(str(start_month) + "-" + str(start_year), "%m-%Y")
        end = datetime.strptime(str(end_month) + "-" + str(end_year), "%m-%Y")
        intervalo_2 = [date for date in intervalo if start <= date <= end]

        # Importando cada archivo de las fechas intermedias
        for date in intervalo_2:
            # Buscando el archivo con el mes y año
            month = date.month
            year = date.year
            patron = os.path.join(path, f"table_clean_*_{month}_{year}.xlsx")

            # Identificando todos los archivos con las coincidencias de mes y año
            archivo = glob.glob(patron)

            # Importando el archivo
            df = pd.read_excel(archivo[0])

            # Creando el key principal
            if year in dict.keys():
                dict[year][month] = df
            else:
                dict[year] = {}
                dict[year][month] = df
                
        return dict

Example of Nested Dictionary

Instead of `df_10_2015` it should have the information for October 2015.

In [ ]:
nested_dict = { '2015' : { '10': 'df_10_2015', 
                       '09': 'df_09_2015'  }, 
              '2016' : { '10': 'df_10_2016', 
                       '09': 'df_09_2016'  }}

2. Execute the following scripts:

In [ ]:
import_data_sbs_banks(10, 2015)

In [ ]:
data = import_data_sbs_banks(10, 2014, 10, 2015)
data

In [ ]:
data.keys()

In [ ]:
data[2014].keys()

In [ ]:
data[2015].keys()